In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import anndata as ad
import scipy as sp
import pandas as pd
import pickle

from tqdm.notebook import tqdm
from scFM_density_estimation.models import *
from sklearn.model_selection import train_test_split

In [2]:
class ConditionalFlowMatchingWithScore(L.LightningModule):
    def __init__(self,
                 input_dim: int,
                 cond_dim: int,
                 hidden_dims: list = [],
                 cond_hidden_dims: list = [16],
                 cond_out_dim: int = 8,
                 lr: float = 1e-3,
                 use_encoder: bool = False,
                 use_ot_sampler: bool = False,
                 ot_method: str = "exact",
                 dropout: float = 0,
                 sigma_min: float = None,
                 ):
        super().__init__()
        self.save_hyperparameters()
        if not use_encoder:
            cond_out_dim = cond_dim
        
        self.cond_encoder = ConditionEncoder(cond_dim, cond_hidden_dims, cond_out_dim, dropout)
        self.vf_mlp = FlowMatchingMLP(input_dim + 1 + cond_out_dim, hidden_dims, input_dim, dropout)
        self.score_mlp = FlowMatchingMLP(input_dim + 1 + cond_out_dim, hidden_dims, input_dim, dropout)
        self.ot_sampler = OTPlanSampler(method=ot_method)
        self.sigma_min = sigma_min
        
        self.use_encoder = use_encoder
        self.use_ot_sampler = use_ot_sampler
        self.cond_dim = cond_dim
        self.lr = lr

    def forward(self, x, t, cond):
        if t.dim() == 0 or t.size()[0] == 1:
            t = t.expand(x.shape[0]).unsqueeze(1)
        elif t.dim() == 1:
            t = t.unsqueeze(1)
            
        if self.use_encoder:
            cond_enc = self.cond_encoder(cond)
        else:
            cond_enc = cond
            
        xtc = torch.cat([x, t, cond_enc], dim=1)
        return self.vf_mlp.mlp(xtc), self.score_mlp.mlp(xtc)

    def shared_step(self, x1, cond):
        device = x1.device
        
        x0 = torch.randn_like(x1).to(device)
        t = torch.rand(x1.shape[0]).unsqueeze(1).to(device)
        eps = torch.randn_like(x1).to(device)
        sigma_t = torch.sqrt(t * (1 - t))
        
        if self.use_ot_sampler:
            x0, x1, cond = self.sample_ot(x0, x1, cond)
        
        if self.sigma_min is None:
            xt = x0 + t * (x1 - x0)
            ut = (1 - 2 * t) / (2 * t * (1 - t) + 1e-8) * sigma_t * eps + x1 - x0
        else:
            xt = x0 + t * (x1 - (1 - self.sigma_min) * x0)
            ut = (1 - 2 * t) / (2 * t * (1 - t) + 1e-8) * sigma_t * eps + x1 - (1 - self.sigma_min) * x0

        pred_ut, pred_score = self(xt + eps * sigma_t, t, cond)
        
        vf_loss = F.mse_loss(pred_ut, ut)
        score_loss = F.mse_loss(2 * sigma_t * pred_score, -eps)
        return vf_loss + score_loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

def prepare_dataset(n, N, cond_dim, locs):
    C = np.random.randint(low=0, high=cond_dim, size=(N))
    X = np.concatenate([np.random.normal(loc=locs[c], scale=1, size=(1, n)) for c in C])

    X_train, X_test, C_train, C_test = train_test_split(X, C, test_size=10_000)

    X_train = torch.tensor(X_train).to("cuda").float()
    C_train = F.one_hot(torch.tensor(C_train).long(), num_classes=cond_dim).to("cuda").float()

    X_test = torch.tensor(X_test).to("cuda").float()
    C_test = F.one_hot(torch.tensor(C_test).long(), num_classes=cond_dim).to("cuda").float()
    return X_train, X_test, C_train, C_test

def initialize_model(n, cond_dim):
    model = ConditionalFlowMatchingWithScore(input_dim=n, hidden_dims=[1024, 1024, 1024], cond_dim=cond_dim,
                                    use_encoder=False, use_ot_sampler=False).to("cuda")
    optimizer = model.configure_optimizers()
    return model, optimizer

def train(batch_size, n_steps, model, optimizer, X, C):
    for k in range(n_steps):
        optimizer.zero_grad()
    
        indices = np.random.choice(range(X.shape[0]), size=batch_size, replace=False)
        loss = model.shared_step(X[indices], C[indices])
        
        loss.backward()
        optimizer.step()
    return model

def div_fn_hutch_trace(u):
    def div_fn(x, eps):
        _, vjpfunc = torch.func.vjp(u, x)
        return (vjpfunc(eps)[0] * eps).sum()

    return div_fn

class NODEWrapper_with_trace_div(torch.nn.Module):
    def __init__(self, model, cond):
        super().__init__()
        self.model = model
        self.cond = cond
        self.div_fn, self.eps_fn = div_fn_hutch_trace, torch.randn_like

    def forward(self, t, x, *args, **kwargs):
        x = x[..., :-1]
        
        def vecfield(y):
            vf, _ = self.model(y.unsqueeze(0), t, self.cond[:1])
            return vf.squeeze()

        div = torch.vmap(self.div_fn(vecfield))(x, self.eps_fn(x))
        dx, _ = self.model(x, t, self.cond)
        return torch.cat([dx, div[:, None]], dim=-1)

class NODEWrapper_with_ratio(torch.nn.Module):
    def __init__(self, model, control, condition):
        super().__init__()
        self.model = model
        self.cond_v = control
        self.cond_u = condition
        self.div_fn, self.eps_fn = div_fn_hutch_trace, torch.randn_like

    def forward(self, t, x, *args, **kwargs):
        x = x[..., :-1]
        
        def vecfield(y):
            ut, _ = self.model(y.unsqueeze(0), t, self.cond_u[:1])
            vt, _ = self.model(y.unsqueeze(0), t, self.cond_v[:1])
            return vt.squeeze() - ut.squeeze()
        
        div = torch.vmap(self.div_fn(vecfield))(x, self.eps_fn(x))
        ut, _ = self.model(x, t, self.cond_u)
        vt, score = self.model(x, t, self.cond_v)
        
        correction_term = torch.linalg.vecdot(vt - ut, score)
        dr = div + correction_term
        
        return torch.cat([ut, dr[:, None]], dim=-1)

class NODEWrapper_with_ratio_tvf(torch.nn.Module):
    def __init__(self, model, control, condition, point):
        super().__init__()
        self.model = model
        self.cond_v = control
        self.cond_u = condition
        self.cond_f = point
        self.div_fn, self.eps_fn = div_fn_hutch_trace, torch.randn_like

    def forward(self, t, x, *args, **kwargs):
        x = x[..., :-1]
        
        def vecfield(y):
            ut, _ = self.model(y.unsqueeze(0), t, self.cond_u[:1])
            vt, _ = self.model(y.unsqueeze(0), t, self.cond_v[:1])
            return vt.squeeze() - ut.squeeze()
        
        div = torch.vmap(self.div_fn(vecfield))(x, self.eps_fn(x))
        ut, score_u = self.model(x, t, self.cond_u)
        vt, score_v = self.model(x, t, self.cond_v)
        ft, _ = self.model(x, t, self.cond_f)
        
        correction_term_u = torch.linalg.vecdot(ft - ut, score_u)
        correction_term_v = torch.linalg.vecdot(vt - ft, score_v)
        dr = div + correction_term_u + correction_term_v
        
        return torch.cat([ft, dr[:, None]], dim=-1)

def evaluate_model(model, data_samples, cond, condition, control, locs):
    device = data_samples.device

    log_condition_true = -0.5 * ((data_samples.cpu().numpy() - np.array(locs[np.argmax(condition)])) ** 2).sum(axis=1) - 0.5 * data_samples.shape[1] * np.log(2 * np.pi)
    log_control_true = -0.5 * ((data_samples.cpu().numpy() - np.array(locs[np.argmax(control)])) ** 2).sum(axis=1) - 0.5 * data_samples.shape[1] * np.log(2 * np.pi)
    log_ratio_true = log_condition_true - log_control_true

    # direct evaluation
    node = NeuralODE(
        NODEWrapper_with_trace_div(model, torch.tensor(condition).float().expand(data_samples.shape[0], cond_dim).to(device)),
        solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4
    )
    
    with torch.no_grad():
        traj = node.trajectory(
            torch.cat([data_samples, torch.zeros(data_samples.shape[0], 1).to(device)], dim=-1),
            t_span=torch.linspace(1, 0, 100).to(device)
        )
    z0, div = traj[-1][:, :-1], traj[-1][:, -1]
    log_condition_hat = (-0.5 * (z0 ** 2).sum(dim=1) - 0.5 * z0.shape[1] * np.log(2 * np.pi) + div).cpu().numpy()
    
    node = NeuralODE(
        NODEWrapper_with_trace_div(model, torch.tensor(control).float().expand(data_samples.shape[0], cond_dim).to(device)),
        solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4
    )
    
    with torch.no_grad():
        traj = node.trajectory(
            torch.cat([data_samples, torch.zeros(data_samples.shape[0], 1).to(device)], dim=-1),
            t_span=torch.linspace(1, 0, 100).to(device)
        )
    z0, div = traj[-1][:, :-1], traj[-1][:, -1]
    log_control_hat = (-0.5 * (z0 ** 2).sum(dim=1) - 0.5 * z0.shape[1] * np.log(2 * np.pi) + div).cpu().numpy()
    
    log_ratio_hat = log_condition_hat - log_control_hat

    # correction term - its own field
    node = NeuralODE(
        NODEWrapper_with_ratio_tvf(model, control=torch.tensor(control).float().expand(data_samples.shape[0], cond_dim).to(device),
                                   condition=torch.tensor(condition).float().expand(data_samples.shape[0], cond_dim).to(device), point=cond),
        solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4
    )
    
    with torch.no_grad():
        traj = node.trajectory(
            torch.cat([data_samples, torch.zeros(data_samples.shape[0], 1).to(device)], dim=-1),
            t_span=torch.linspace(1, 0, 100).to(device)
        )
    z0, ratio = traj[-1][:, :-1], traj[-1][:, -1]
    log_ratio_hat_v3 = -ratio.cpu().numpy()
    
    return log_ratio_true, log_ratio_hat, log_ratio_hat_v3

In [26]:
n = 10
N = 30_000
cond_dim = 4
batch_size = 512
n_steps = 50_000
n_tries = 10

condition = np.array([0, 1, 0, 0])
control = np.array([1, 0, 0, 0])

In [27]:
# easy
# locs = [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 2, 0, 2, 2, 2, 0, 0, 2], [2, 2, 0, 0, 2, 2, 2, 0, 0, 2], [-1, -1, 3, -1, 3, 3, 3, -1, -1, 3]]

# difficult
locs = [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 0, 0, 0, 0, 2, 2, 2, 2], [2, 0, 0, 2, 0, 0, 0, 2, 0, 0], [3, 3, 3, 3, -1, 3, 3, 3, 3, -1]]

In [28]:
sigmas = [0]
# names = ["true", "direct", "ct condition", "ct own", "ct control"]
names = ["true", "direct", "ct own"]
results = {sigma: [] for sigma in sigmas}

In [29]:
X_train, X_test, C_train, C_test = prepare_dataset(n, N, cond_dim, locs)

mask_condition = np.all(C_test.cpu().numpy() == condition, axis=1)
mask_control = np.all(C_test.cpu().numpy() == control, axis=1)
mask_both = mask_condition | mask_control

for sigma_min in sigmas:
    tmp_results = {name: np.array([]) for name in names}
    for _ in tqdm(range(n_tries)):
        model, optimizer = initialize_model(n, cond_dim)
        model = train(batch_size, n_steps, model, optimizer, X_train, C_train)
        log_ratios = evaluate_model(model, X_test, C_test, condition, control, locs)

        for i, name in enumerate(names):
            if tmp_results[name].shape[0] == 0:
                tmp_results[name] = log_ratios[i].reshape(-1, 1)
            else:
                tmp_results[name] = np.concatenate([tmp_results[name], log_ratios[i].reshape(-1, 1)], axis=1)

    results[sigma_min] = tmp_results

  0%|          | 0/10 [00:00<?, ?it/s]

In [30]:
results["mask_condition"] = mask_condition
results["mask_control"] = mask_control
results["mask_both"] = mask_both

In [31]:
with open("./table_results/really_easy_results_lsm_1.pkl", "wb") as f:
    pickle.dump(results, f)

In [32]:
with open("./table_results/really_easy_results_lsm_1.pkl", "rb") as f:
    results = pickle.load(f)

mask_condition = results["mask_condition"]
mask_control = results["mask_control"]
mask_both = results["mask_both"]

In [33]:
def mean_confidence_interval(x, confidence=0.95):
    m, se = np.mean(x), sp.stats.sem(x)
    h = se * sp.stats.t.ppf((1 + confidence) / 2., x.shape[0] - 1)
    return np.round(m - h, 2), np.round(m, 2), np.round(m + h, 2)

In [34]:
processed_results = {e: {sigma: {name: np.array([]) for name in names} for sigma in sigmas} for e in ["sum", "mean"]}

for sigma in sigmas:
    for name in names:
        tmp_sum = np.sum(results[sigma][name][mask_condition], axis=0)
        tmp_mean = np.mean(results[sigma][name][mask_condition], axis=0)
        
        processed_results["sum"][sigma][name] = mean_confidence_interval(tmp_sum)
        processed_results["mean"][sigma][name] = mean_confidence_interval(tmp_mean)

In [35]:
df_mean = pd.DataFrame(processed_results["mean"]).transpose()
df_mean

,true,direct,ct own
0,"(5.04, 5.04, 5.04)","(4.76, 5.14, 5.52)","(2.23, 2.39, 2.55)"


In [14]:
# difficult example
df_mean

,true,direct,ct own
0,"(20.0, 20.0, 20.0)","(18.13, 19.77, 21.4)","(8.69, 9.07, 9.45)"


In [25]:
# "easy" example
df_mean

,true,direct,ct own
0,"(20.25, 20.25, 20.25)","(18.6, 19.77, 20.93)","(8.63, 8.92, 9.21)"


In [36]:
# really easy example
df_mean

,true,direct,ct own
0,"(5.04, 5.04, 5.04)","(4.76, 5.14, 5.52)","(2.23, 2.39, 2.55)"
